#### Figure 2F, 3F glms

In [ ]:

%reload_ext autoreload
%autoreload 2

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.pyplot import cm
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import seaborn as sns
import matplotlib as mpl

from itertools import cycle

from paths import DATA_DIR, fig_dir

from sklearn.metrics import confusion_matrix, accuracy_score, roc_curve, roc_auc_score, precision_recall_curve, precision_score, recall_score, auc, f1_score, average_precision_score, confusion_matrix, balanced_accuracy_score
from sklearn.model_selection import train_test_split, KFold
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE
from scipy.stats import norm
from statsmodels.stats.proportion import proportion_confint, proportions_ztest
from scipy.stats import wilcoxon

In [ ]:
from spyglass.common import Session

In [ ]:
# custom schema
from find_my_data import *
from alison_decoding import ClusterlessAcausalResultsSummary
from fig_helpers import *


In [ ]:
set_figure_defaults()

fig_path = fig_dir('figs26')
if not os.path.exists(fig_path):
    os.makedirs(fig_path)

custom_colors_by_rat = iter(cm.tab20b([0,.8, .85, .1, .05]))

save_fig = False

In [ ]:
# params
position_info_param_name='default_decoding'
remove_hpd_timepoints = True
hpd_percent = 50
hpd_threshold = 50
require_nonlocal_by_segment = False
remove_low_speed_timepoints = True
head_speed_threshold = 10

# data loading info
out_path = f'{DATA_DIR}/big_df_pkls/'
today_now = '20240212'

subject_ids = ['senor', 'chimi', 'j16', 'wilbur', 'peanut']
custom_colors_by_rat = iter(cm.tab20b([0,.8, .85, .1, .05]))


### load data

In [ ]:
# load behavior and decoding days of data, crosscheck nwbs
big_dfs = {}
for subject_id in subject_ids:
    try:
        big_dfs[subject_id] = pd.read_pickle(out_path+subject_id.lower()+'_big_df_RL_deltaq_stable'+today_now+'.pkl')
    except Exception as e:
        print('exception',e)

stable_nwbs = {}
clusterless_nwbs = {}
stable_clusterless_nwbs = {}
for subject_id in subject_ids:
    stable_nwbs[subject_id] = list( (Session & {'session_description LIKE "Spatial bandit task (regular)"'}
                                             & {"subject_id": subject_id}).fetch('nwb_file_name') )
    clusterless_nwbs[subject_id] = list(np.unique((ClusterlessAcausalResultsSummary()
                                                   & spatial_bandit_query_by_rat(rat_list=[subject_id])).fetch('nwb_file_name')))
    if subject_id == 'j16':
        stable_nwbs['j16'].remove('mediumnwb20230802_.nwb')
    if subject_id == 'chimi':
        stable_nwbs['chimi'].remove('chimi20200216_new_.nwb')
    if subject_id == 'senor':
        stable_nwbs['senor'].remove('senor20201030_.nwb')
    stable_clusterless_nwbs[subject_id] = [nwb for nwb in clusterless_nwbs[subject_id] if nwb in stable_nwbs[subject_id]]
print(stable_clusterless_nwbs)


In [ ]:
is_mapped_seg_a_leaf_map = {0:False, 1:True, 2:True, 3:False, 4:True, 5:True, 6:False, 7:True, 8:True}
segs_to_patch_map = {0:1, 1:1, 2:1, 3:2, 4:2, 5:2, 6:3, 7:3, 8:3}
p_rew_cols = [f"p_rew_leaf{i}" for i in [1,2,3,4,5,6]]

# Restrict big df to those with clusterless data, and agument for related analyses, ensure no 100all 50all
all_rat_big_dfs_stable = {}
for subject_id in subject_ids:
    df = big_dfs[subject_id]
    df_stable = df[df['nwb_file_name'].isin(stable_clusterless_nwbs[subject_id])]
    df_stable['is_actual_seg_mapped_a_leaf'] = df_stable[['actual_segment_mapped']].applymap(is_mapped_seg_a_leaf_map.get)
    df_stable['is_mental_seg_mapped_a_leaf'] = df_stable[['mental_segment_mapped']].applymap(is_mapped_seg_a_leaf_map.get)
    df_stable['mental_patch_mapped'] = df_stable[['mental_segment_mapped']].applymap(segs_to_patch_map.get)
    all_rat_big_dfs_stable[subject_id] = df_stable[~df_stable[p_rew_cols].eq(df_stable['p_rew_leaf1'], axis=0).all(axis=1)]


In [ ]:
# Filter data to segments of interest 
hpd_percent = 50 # or 95
hpd_thresh_cm = 50 # 50
use_abs_ahbeh_thresh = False
abs_ahbeh_thresh_cm = 10
quantile = .9

big_dfs_firstlast_nonlocal_incljump_grouped = {}
for subject_id in subject_ids:
    big_df = all_rat_big_dfs_stable[subject_id]
    
    # limit to first or last track segment, add any hpd and ahbeh restrictions for quality control
    big_df_firstlast = big_df[np.logical_and(
                                    np.logical_or(big_df['is_first_seg_of_trial']==True,
                                                  big_df['is_last_seg_of_trial']==True),
                                    big_df[f'spatial_coverage_{hpd_percent}_hpd']<hpd_thresh_cm,
                                    )]
    if use_abs_ahbeh_thresh:
        big_df_firstlast = big_df_firstlast[big_df_firstlast['abs_ahead_behind_distance']>=abs_ahbeh_thresh_cm]
    
    big_df_firstlast_nonlocal = big_df_firstlast[big_df_firstlast['nonlocal_by_segment']==True]
    
    # find nonlocal stay and switch consistent content
    big_df_firstlast_nonlocal['is_mental_seg_leaf_in_patch_or_elsewhere'] = np.logical_and(
                                                                    big_df_firstlast_nonlocal['nonlocal_by_patch']==False,
                                                                    big_df_firstlast_nonlocal['is_mental_seg_mapped_a_leaf']==True)
    big_df_firstlast_nonlocal['is_mental_seg_elsewhere_or_leaf_in_patch'] = ~big_df_firstlast_nonlocal['is_mental_seg_leaf_in_patch_or_elsewhere']
    
    # now do all the groupings to calc things for first/final seg data
    # these calcultaions are only during nonlocal by seg times, not full trial time
    big_df_grouped = big_df_firstlast_nonlocal.groupby(
            by=['nwb_file_name', 'epoch_number', 'trial_number_by_epoch', 'stem_switch', 'stem', 'leaf', 'stemchoice', 'reward', 'is_first_seg_of_trial',
                'trials_from_prior_switch', 'trials_from_next_switch','try_bout_idx']
        ).apply(
            lambda x_df: pd.Series({
                'prop_elsewhere_vs_neighbor_leaf': len(x_df[x_df['is_mental_seg_elsewhere_or_leaf_in_patch']]) / len(x_df),
                'len_elsewhere': len(x_df[x_df['is_mental_seg_elsewhere_or_leaf_in_patch']]),
                'len_neighbor_leaf': len(x_df[x_df['is_mental_seg_leaf_in_patch_or_elsewhere']]),
                'ahbeh_mean':x_df['abs_ahead_behind_distance'].mean(),
                'ahbeh_mean_neighbor_leaf':x_df[x_df['is_mental_seg_leaf_in_patch_or_elsewhere']]['abs_ahead_behind_distance'].mean(),
                'ahbeh_mean_elsewhere':x_df[x_df['is_mental_seg_elsewhere_or_leaf_in_patch']]['abs_ahead_behind_distance'].mean(),
                'ahbeh_max':x_df['abs_ahead_behind_distance'].max(),
                'ahbeh_max_neighbor_leaf':x_df[x_df['is_mental_seg_leaf_in_patch_or_elsewhere']]['abs_ahead_behind_distance'].max(),
                'ahbeh_max_elsewhere':x_df[x_df['is_mental_seg_elsewhere_or_leaf_in_patch']]['abs_ahead_behind_distance'].max(),
                f'ahbeh_quantile{int(100*quantile)}':x_df['abs_ahead_behind_distance'].quantile(q=quantile),
                f'len_nonlocal': len(x_df),
            })
        ).reset_index()
    big_dfs_firstlast_nonlocal_incljump_grouped[subject_id] = big_df_grouped


### functions

In [ ]:
# Functions


def plot_results_by_subject_then_model_formatted(results, ci, first_seg, final_seg, threshold, p_values=None, save_fig=False, model_labels=None, connect_models=True,
                                                figwidth=6,figheight=4,capsize=5, markersize=40, edgecolors='face',linewidth=1):
    # Custom color palette for each subject, using specific values you provided
    color_palette = cm.tab20b([0, 0.8, 0.85, 0.1, 0.05])
    
    # Prepare markers and model labels if not provided
    model_names = list(results.keys())
    markers = ['o', 's', '^']  # Circle, Square, Triangle for models
    if model_labels is None:
        model_labels = model_names # Default labels if none provided
    
    subject_ids = list(results[model_names[0]].keys())  # Assumes all models have the same subjects
    
    fig, ax = plt.subplots(figsize=(figwidth,figheight))
    subject_positions = {}

    for model_idx, model_name in enumerate(model_names):
        for subject_idx, (subject_id, data) in enumerate(results[model_name].items()):
            proportion = data['proportion_correct']
            ci_lower, ci_upper = data['ci_lower'], data['ci_upper']
            x_pos = subject_idx + (model_idx - len(model_names) / 2) * 0.2  # Adjust model positioning around subject
            
            # Error bars and scatter plot with the specified color palette
            ax.errorbar(x_pos, proportion, yerr=[[proportion - ci_lower], [ci_upper - proportion]],
                        fmt=markers[model_idx], capsize=capsize, color=color_palette[subject_idx],zorder=10)
#                         label=f'_{model_labels[model_idx] if subject_idx == 0 else None}')
            ax.scatter(x_pos, proportion, marker=markers[model_idx], s=markersize, color=color_palette[subject_idx],
                       edgecolors=edgecolors,linewidth=linewidth,
                       label=model_labels[model_idx] if subject_idx == 0 else None,zorder=1000)
            
            # Store positions for line drawing
            if subject_id not in subject_positions:
                subject_positions[subject_id] = []
            subject_positions[subject_id].append((x_pos, proportion))

    # Draw lines for each subject
    if connect_models:
        for subject_id, positions in subject_positions.items():
            x_values, y_values = zip(*positions)
            ax.plot(x_values, y_values, linestyle='-', linewidth=1, color=color_palette[list(subject_positions.keys()).index(subject_id)], alpha=0.5, zorder=0)

    # Setting labels and titles
    ax.set_ylabel('Proportion of Correct\nClassifications')
    ax.set_title(f'Proportion Correct by Subject and Model, CI {ci}\nFirst: {first_seg}, Final: {final_seg}', fontsize=4) #y=1.05)
    chance_line = ax.axhline(0.5, linestyle='--', color='grey', label='Chance')

    # Collecting handles and labels and reordering them
    handles, labels = ax.get_legend_handles_labels()
    # Move the 'Chance' line to the end of the legend
    chance_index = labels.index('Chance')
    chance_handle = handles.pop(chance_index)
    labels.pop(chance_index)
    handles.append(chance_handle)
    labels.append('Chance')
    
    ax.legend(handles, labels, bbox_to_anchor=(1.01, .9), loc='upper left', frameon=False)

    plt.ylim(0.48, 1)
    plt.xlim(-0.5, len(subject_ids) - 0.5)
    sns.despine(offset=5)
    
    plt.xticks(range(len(subject_ids)), [f'Rat {id[0].upper()}' for id in subject_ids],)# rotation=45)
#     plt.tight_layout()
    if save_fig:
        fig_name = f'fmt2_each_rat_accuracy_scatter_nopvals_compare_model_bymodel_modelmarkers_ci{ci}_ztest_first{first_seg}_final{final_seg}_thresh{threshold}_connectmodels{connect_models}_edgecolors{edgecolors}_cap{capsize}_markersize{markersize}_lw{linewidth}_h{figheight}_w{figwidth}'
        plt.savefig(f'{fig_path}{fig_name}.pdf', format='pdf', bbox_inches="tight", pad_inches=.5)    
    plt.show()


def process_data(model_data, ci=.95): # data_dicts
    results = {}
    for model_name, y_tests, predictions in model_data:
        model_results = {}
        for subject_id, actuals in y_tests.items():
            preds = np.array(predictions[subject_id])
            binarized_preds = preds >= 0.5
            correct = binarized_preds == actuals
            proportion_correct = np.mean(correct)
            ci_lower, ci_upper = proportion_confint(sum(correct), len(correct), alpha=1-ci, method='normal')
            model_results[subject_id] = {'proportion_correct': proportion_correct, "ci_lower": ci_lower, 'ci_upper': ci_upper, 'correct_incorrect':correct}
        results[model_name] = model_results
    return results

def process_data_accuracies_cis(model_data, ci=.95, use_balanced_accuracy=False):
    '''
    For balanced or normal accuracy with ci on the proportion
    '''
    results = {}
    for model_name, y_tests, predictions in model_data:
        model_results = {}
        for subject_id, actuals in y_tests.items():
            preds = np.array(predictions[subject_id])
            binarized_preds = preds >= 0.5
            correct = binarized_preds == actuals
            if use_balanced_accuracy == False:
                proportion_correct = np.mean(correct) # accuracy score
                ci_lower, ci_upper = proportion_confint(sum(correct), len(correct), alpha=1-ci, method='normal')
            elif use_balanced_accuracy == True:
                balanced_accuracy = balanced_accuracy_score(actuals, binarized_preds)
                proportion_correct = balanced_accuracy # a little weird to calculate ci on proportion for balanced accuracy though
                nobs = len(actuals)
                count = np.round(balanced_accuracy*nobs)
                ci_lower, ci_upper = proportion_confint(count, nobs, alpha=1-ci, method='normal')
            model_results[subject_id] = {'proportion_correct': proportion_correct, "ci_lower": ci_lower, 'ci_upper': ci_upper, 'correct_incorrect':correct}
        results[model_name] = model_results
    return results # usually normal accuracy, but can be balanced accuracy if you want

def plot_switch_roc_stratify_resample_kfold(data_dict, predictors, plot_per_subject, subject_ids, save_fig, first_seg, final_seg, is_inner_merge=False):
    family = sm.families.Binomial()
    y_tests = {subject_id: [] for subject_id in subject_ids}
    predictionss = {subject_id: [] for subject_id in subject_ids}
    d_primes = {subject_id: [] for subject_id in subject_ids}
    f1_scores = {subject_id: [] for subject_id in subject_ids}

    for subject_id in subject_ids:
        all_fpr = []
        all_tpr = []
        all_roc_auc = []
        all_recall = []
        all_precision = []
        all_pr_auc = []
        all_f1_scores = []
        all_d_prime = []
        
        # subject_id = 'j16'
        full_data = data_dict[subject_id][data_dict[subject_id][predictors].notna().all(axis=1)] # find the rows of data without nans
        X = full_data[predictors] # x only
        y = full_data['stem_switch_float'] # y only
        print(f'Len of full data: {len(X)}')
        
        
        kf = KFold(n_splits=5, shuffle=True, random_state = 42)
        
        for train_index, test_index in kf.split(X):
            # Split data into training and test sets
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train, y_test = y.iloc[train_index], y.iloc[test_index]
            print(f'Len of train: {len(X_train)} and test, which is what predictions are on: {len(X_test)}')
            
            y_tests[subject_id].extend(y_test)
            
            # Combine X_train and y_train for upsampling
            train_data = pd.concat([X_train, y_train], axis=1)

            ## upsample minority class after splitting into train test sets to avoid leakage
            switch_train = train_data[train_data['stem_switch_float']==1] # split training data into stay and switch trials, majority and minority classes 
            stay_train = train_data[train_data['stem_switch_float']==0] # note this has both x and y info in it
            switch_train_upsampled = resample(switch_train, replace=True, n_samples=len(stay_train), random_state=42) # upsampel the minority switch class

            # to be able to concatenate them i need to reindex so there aren't duplicate indices
            stay_train_resetidx = stay_train.reset_index(drop=True)
            switch_train_upsampled_resetidx = switch_train_upsampled.reset_index(drop=True)
            # this will use the reindexed data to make one main upsampled training dataset with both x and y info
            train_data_upsampled = pd.concat([stay_train_resetidx, switch_train_upsampled_resetidx], axis=0) #concatenate vertically now not horizontally

            # split into x and y
            X_train_upsampled = train_data_upsampled.drop('stem_switch_float', axis=1) # this isnt being used rn
            y_train_upsampled = train_data_upsampled['stem_switch_float'] # and this is only being used to print stuff out
            print(f'Len of train upsampled {len(train_data_upsampled)}')

            formula = f"stem_switch_float ~ {' + '.join(predictors)}"
            model_fit = smf.glm(formula=formula, data=train_data_upsampled, family=family).fit()
            print(f'\n{subject_id}\n', model_fit.summary())

            # Make predictions and calculate metrics for each fold
            predictions = model_fit.predict(X_test)
            predictionss[subject_id].extend(predictions)
            predictions_binary = np.where(predictions > .5, 1, 0)
            
#             print(f'Switches in training set: {y_train.sum()}, out of {len(y_train)}, so {round(y_train.sum()/len(y_train),2)}.')
#             print(f'Switches in upsampled training set: {y_train_upsampled.sum()}, out of {len(y_train_upsampled)}, so {round(y_train_upsampled.sum()/len(y_train_upsampled),2)}.')
#             print(f'Switches in test set: {y_test.sum()}, out of {len(y_test)}, so {round(y_test.sum()/len(y_test),2)}.')
            predictions_binary = np.where(predictions >.5, 1, 0)
            conf_matrix = confusion_matrix(y_test,predictions_binary)
            accuracy = accuracy_score(y_test, predictions_binary)
#             print(f'Confusion matrix: \n {conf_matrix}.')
#             print(f'Accuracy: {round(accuracy,4)}.')
            
            # Calculate and store ROC and PR curve metrics
            fpr, tpr, _ = roc_curve(y_test, predictions)
            roc_auc = roc_auc_score(y_test, predictions)
            all_fpr.append(fpr)
            all_tpr.append(tpr)
            all_roc_auc.append(roc_auc)

            precision, recall, _ = precision_recall_curve(y_test, predictions)
            pr_auc = auc(recall, precision)
            all_recall.append(recall)
            all_precision.append(precision)
            all_pr_auc.append(pr_auc)

            # Calculate and store F1 Score and d'
            f1 = f1_score(y_test, predictions_binary)
            all_f1_scores.append(f1)
            d_prime = get_dprime(conf_matrix)
            all_d_prime.append(d_prime)
        
        # aggregate data across fold within rat, and plot
        if plot_per_subject:
            fig, axes = plt.subplots(ncols=2, figsize=(20, 6))
            # Plot aggregated ROC curves
            for i in range(5):
                axes[0].plot(all_fpr[i], all_tpr[i], label=f'Fold {i+1} ROC (AUC = {all_roc_auc[i]:.2f})')

            # Plot aggregated PR curves
            for i in range(5):
                axes[1].plot(all_recall[i], all_precision[i], label=f'Fold {i+1} PR (AUC = {all_pr_auc[i]:.2f})')

            # Add details to the plots
            axes[0].plot([0, 1], [0, 1], color='grey', linestyle='--')
            axes[0].set_xlabel('FPR')
            axes[0].set_ylabel('TPR')
            axes[0].set_title('ROC (5-Fold CV)')
            axes[0].legend(loc='upper left', bbox_to_anchor=(1,1), fontsize=12)

            axes[1].set_xlabel('Recall')
            axes[1].set_ylabel('Precision')
            axes[1].set_title('PR (5-Fold CV)')
            axes[1].legend(loc='upper left', bbox_to_anchor=(1,1), fontsize=12)

            plt.subplots_adjust(wspace=1)
            plt.suptitle(f'{subject_id}: {formula}\nfirstseg: {first_seg}, finalseg: {final_seg}', fontsize=12, y=1.05)
            if save_fig:
                formulaname = ''.join(e for e in formula if e.isalnum())
                fig_name = f'5fold_{subject_id}_roc_pr_firstseg{first_seg}_finalseg{final_seg}_{formulaname}'
                if is_inner_merge:
                    fig_name = f'innermerge{is_inner_merge}_{fig_name}'
                plt.savefig(f'{fig_path}{fig_name}.pdf', format='pdf', bbox_inches="tight", pad_inches=.5)       
        d_primes[subject_id].extend(all_d_prime)
        f1_scores[subject_id].extend(all_f1_scores)
    return predictionss, y_tests, d_primes, f1_scores

def get_dprime(conf_matrix):
    tn, fp, fn, tp = conf_matrix.ravel()
    hr = tp / (tp + fn) # hit rate
    far = fp / (fp + tn) # flase alarm rate
    # adjust in case of 0s and 1s edge cases before z scoring
    n_signal_present = tp + fn  # total n of actual positives
    n_signal_absent = tn + fp  # total n of actual negatives
    if hr == 0:
        hr = 0.5 / n_signal_present
    elif hr == 1:
        hr = (n_signal_present - 0.5) / n_signal_present
    if far == 0:
        far = 0.5 / n_signal_absent
    elif far == 1:
        far = (n_signal_absent - 0.5) / n_signal_absent
    # Convert HR and FAR to Z-scores
    z_hr = norm.ppf(hr)
    z_far = norm.ppf(far)
    # Calculate d'
    d_prime = z_hr - z_far
    return d_prime

def calculate_f2_glm_p_values(results): # assumes 3 models from ms only
#     p_values = {}
    model_names = list(results.keys())
    subjects = list(results[model_names[0]].keys())
    for subject_id in subjects:
        print(f'\n{subject_id}')
        model_pairs = [[model_names[0], model_names[1]],
                      [model_names[1], model_names[2]],
                      [model_names[0],model_names[2]]]
        for model_pair in model_pairs:
            counts = [np.sum(results[model_name][subject_id]['correct_incorrect']) for model_name in model_pair] 
            nobs = np.array([len(results[model_name][subject_id]['correct_incorrect']) for model_name in model_pair] ) #[len(y_tests_a), len(y_tests_b)])
            
            stat, p_value = proportions_ztest(counts, nobs)
#             p_values[subject_id] = p_value
#             stats[subject_id] = stat
#             nobs[subject_id] = nobs
#             counts[subject_id] = counts
            print(f'COMPARE {model_pair} proportions z test')
            print(f'counts: {counts}, nobs: {nobs}, stats: {stat}, pvals: {p_value}')
        #also compare models to chance .5
        for model_name in model_names:
            print(f'COMPARE {model_name} to chance 0.5, proportions z test')
            counts = np.sum(results[model_name][subject_id]['correct_incorrect'])
            nobs = len(results[model_name][subject_id]['correct_incorrect'])
            proportions_ztest(counts, nobs, value=.5)
            print(f'counts: {counts}, nobs: {nobs}, stats: {stat}, pvals: {p_value}, compared_to_value: {.5}')
    return #p_values, stats, nobs, counts

# associated stats
def calculate_f3_glm_p_values(results): # assumes 2 models from ms only
#     p_values = {}
    model_names = list(results.keys())
    subjects = list(results[model_names[0]].keys())
    for subject_id in subjects:
        print(f'\n{subject_id}')
#         model_pairs = [[model_names[0], model_names[1]]]
#                       [model_names[1], model_names[2]]]
#                       [model_names[0],model_names[2]]]
#         for model_pair in model_pairs:
        counts = [np.sum(results[model_name][subject_id]['correct_incorrect']) for model_name in model_names] 
        nobs = np.array([len(results[model_name][subject_id]['correct_incorrect']) for model_name in model_names] ) #[len(y_tests_a), len(y_tests_b)])

        stat, p_value = proportions_ztest(counts, nobs)
#             p_values[subject_id] = p_value
#             stats[subject_id] = stat
#             nobs[subject_id] = nobs
#             counts[subject_id] = counts
        print(f'COMPARE {model_names} proportions z test')
        print(f'counts: {counts}, nobs: {nobs}, stats: {stat}, pvals: {p_value}')
        #also compare models to chance .5
        for model_name in model_names:
            print(f'COMPARE {model_name} to chance 0.5, proportions z test')
            counts = np.sum(results[model_name][subject_id]['correct_incorrect'])
            nobs = len(results[model_name][subject_id]['correct_incorrect'])
            proportions_ztest(counts, nobs, value=.5)
            print(f'counts: {counts}, nobs: {nobs}, stats: {stat}, pvals: {p_value}, compared_to_value: {.5}')
    #     return p_values, stats, nobs, counts


### reorganize and plot data

In [ ]:
# data org
# make nice looking plots first for First seg content ROC, PR, and a few dif versions of the variables of interest
data_dict_first = {}
data_dict_last = {}

for subject_id in subject_ids:
    data = big_dfs_firstlast_nonlocal_incljump_grouped[subject_id]
    data['stem_switch_float'] = data['stem_switch'].astype(float)
    data['ahbeh_max_log_scaled'] = np.log(data['ahbeh_max'])/5
    data_first = data[data['is_first_seg_of_trial']==True]
    data_last = data[data['is_first_seg_of_trial']==False]
    data_dict_first[subject_id] = data_first
    data_dict_last[subject_id] = data_last

# this one is within bout rather than only within day epoch. it also adds on integrating ahbeh max rather than just prop content
n_shifts = 10
for is_first_seg in [True,False]:
    if is_first_seg:
        data_dict = data_dict_first
    else:
        data_dict = data_dict_last
    for subject_id in subject_ids:
        grouped = data_dict[subject_id].groupby(["nwb_file_name","epoch_number", "try_bout_idx"])
        for span in range(2,n_shifts+1): #span includes current row, so span 2 is current trial and 1 ago summed/avgd
            if is_first_seg:
                #data_dict[subject_id][f"prop_elsewhere_vs_neighbor_leaf_rolling_sum{span}"] = grouped['prop_elsewhere_vs_neighbor_leaf'].rolling(span, min_periods=min_periods).sum()
                min_periods = 1
                data_dict[subject_id][f"prop_elsewhere_vs_neighbor_leaf_rolling_mean{span}_mp{min_periods}_bout"] = grouped['prop_elsewhere_vs_neighbor_leaf'].rolling(
                    span, min_periods=min_periods).mean().values                
                data_dict[subject_id][f"ahbeh_max_log_scaled_rolling_mean{span}_mp{min_periods}_bout"] = grouped['ahbeh_max_log_scaled'].rolling(
                    span, min_periods=min_periods).mean().values
                data_dict[subject_id][f"prop_elsewhere_vs_neighbor_leaf_rolling_mean{span}_mp{min_periods}_bout_excludeswitch"] = grouped['prop_elsewhere_vs_neighbor_leaf'].shift(1).rolling(
                    span, min_periods=min_periods).mean().values
                data_dict[subject_id][f"ahbeh_max_log_scaled_rolling_mean{span}_mp{min_periods}_bout_excludeswitch"] = grouped['ahbeh_max_log_scaled'].shift(1).rolling(
                    span, min_periods=min_periods).mean().values
                #sum instead of avg
                data_dict[subject_id][f"prop_elsewhere_vs_neighbor_leaf_rolling_sum{span}_mp{min_periods}_bout"] = grouped['prop_elsewhere_vs_neighbor_leaf'].rolling(
                    span, min_periods=min_periods).sum().values                
                data_dict[subject_id][f"ahbeh_max_log_scaled_rolling_sum{span}_mp{min_periods}_bout"] = grouped['ahbeh_max_log_scaled'].rolling(
                    span, min_periods=min_periods).sum().values
                data_dict[subject_id][f"prop_elsewhere_vs_neighbor_leaf_rolling_sum{span}_mp{min_periods}_bout_excludeswitch"] = grouped['prop_elsewhere_vs_neighbor_leaf'].shift(1).rolling(
                    span, min_periods=min_periods).sum().values
                data_dict[subject_id][f"ahbeh_max_log_scaled_rolling_sum{span}_mp{min_periods}_bout_excludeswitch"] = grouped['ahbeh_max_log_scaled'].shift(1).rolling(
                    span, min_periods=min_periods).sum().values
            else: # for final seg use last two rather than current and prior because still wan tto only consider 2 trials but not the post choice point info
                #data_dict[subject_id][f"prop_elsewhere_vs_neighbor_leaf_rolling_sum{span}"] = grouped['prop_elsewhere_vs_neighbor_leaf'].shift(1).rolling(span, min_periods=min_periods).sum()
                min_periods = 1
                data_dict[subject_id][f"prop_elsewhere_vs_neighbor_leaf_rolling_mean{span}_mp{min_periods}_bout"] = grouped['prop_elsewhere_vs_neighbor_leaf'].shift(1).rolling(
                    span, min_periods=min_periods).mean().values
                data_dict[subject_id][f"ahbeh_max_log_scaled_rolling_mean{span}_mp{min_periods}_bout"] = grouped['ahbeh_max_log_scaled'].shift(1).rolling(
                     span, min_periods=min_periods).mean().values
                data_dict[subject_id][f"prop_elsewhere_vs_neighbor_leaf_rolling_sum{span}_mp{min_periods}_bout"] = grouped['prop_elsewhere_vs_neighbor_leaf'].shift(1).rolling(
                    span, min_periods=min_periods).sum().values
                data_dict[subject_id][f"ahbeh_max_log_scaled_rolling_sum{span}_mp{min_periods}_bout"] = grouped['ahbeh_max_log_scaled'].shift(1).rolling(
                     span, min_periods=min_periods).sum().values

n_shifts = 10
for is_first_seg in [True,False]:
    if is_first_seg:
        data_dict = data_dict_first
    else:
        data_dict = data_dict_last
    for subject_id in subject_ids:
        data_dict[subject_id]['try_bout_idx_1_ago'] = data_dict[subject_id]['try_bout_idx'].shift(1)
        grouped = data_dict[subject_id].groupby(["nwb_file_name","epoch_number", "try_bout_idx_1_ago"])
        for n in range(1,n_shifts+1):
            data_dict[subject_id][f"reward_{n}_ago_bout"] = grouped['reward'].shift(n)
            data_dict[subject_id][f'ahbeh_max_log_scaled_{n}_ago_bout'] = grouped['ahbeh_max_log_scaled'].shift(n)
            data_dict[subject_id][f'prop_elsewhere_vs_neighbor_leaf_{n}_ago_bout'] = grouped['prop_elsewhere_vs_neighbor_leaf'].shift(n)
        for subject_id in subject_ids: # or just within epoch
            grouped = data_dict[subject_id].groupby(["nwb_file_name","epoch_number"])
            for n in range(1,n_shifts+1): 
                data_dict[subject_id][f"ahbeh_max_log_scaled_{n}_ago"] = grouped['ahbeh_max_log_scaled'].shift(n)
                data_dict[subject_id][f"prop_elsewhere_vs_neighbor_leaf_{n}_ago"] = grouped['prop_elsewhere_vs_neighbor_leaf'].shift(n)
                

variables = ["len_elsewhere", "len_neighbor_leaf", "ahbeh_max_neighbor_leaf", "ahbeh_max_elsewhere", "ahbeh_max"]
for is_first_seg in [True,False]:
    if is_first_seg:
        data_dict = data_dict_first
    else:
        data_dict = data_dict_last
    for subject_id in subject_ids:
        grouped = data_dict[subject_id].groupby(["nwb_file_name","epoch_number", "try_bout_idx_1_ago"])
        for variable in variables:
            data_dict[subject_id][f'{variable}_1_ago_bout'] = grouped[f'{variable}'].shift(1)
            



In [ ]:
is_first_seg_of_trial = True

first_seg = is_first_seg_of_trial
final_seg = not is_first_seg_of_trial

is_inner_merge=False

predictors_a = ['prop_elsewhere_vs_neighbor_leaf']
predictors_c = ['prop_elsewhere_vs_neighbor_leaf_1_ago_bout']
predictors_e = ['prop_elsewhere_vs_neighbor_leaf_rolling_mean2_mp1_bout']

if (first_seg == True): #& (final_seg == False):
    data_dict = data_dict_first
elif (first_seg == False) & (final_seg == True):
    data_dict = data_dict_last

plot_per_subject=False
ci = 0.95
threshold = 0.5

predictionss_a, y_tests_a, d_primes_a, f1_scores_a = plot_switch_roc_stratify_resample_kfold(data_dict, predictors_a, plot_per_subject, subject_ids, save_fig, first_seg, final_seg, is_inner_merge=is_inner_merge)
predictionss_c, y_tests_c, d_primes_c, f1_scores_c = plot_switch_roc_stratify_resample_kfold(data_dict, predictors_c, plot_per_subject, subject_ids, save_fig, first_seg, final_seg, is_inner_merge=is_inner_merge)
predictionss_e, y_tests_e, d_primes_e, f1_scores_e = plot_switch_roc_stratify_resample_kfold(data_dict, predictors_e, plot_per_subject, subject_ids, save_fig, first_seg, final_seg, is_inner_merge=is_inner_merge)


data_dicts = [(predictors_c[0], y_tests_c, predictionss_c),(predictors_a[0], y_tests_a, predictionss_a),(predictors_e[0], y_tests_e, predictionss_e),]



In [ ]:
results = process_data(data_dicts, ci)
# balanced_results = process_data_accuracies_cis(data_dicts, ci, use_balanced_accuracy=True)


model_labels = [f'Stay Trial before\nSwitch Trial', f'Switch Trial\nbefore Choice', f'Switch Trial and\nPrior Stay Trial']



In [ ]:
figwidth = (TWO_COLUMN/2)*1/GOLDEN_RATIO
figheight = TWO_COLUMN/2
capsize = 5
markersize = 60
edgecolors = 'face' #'white'
linewidth=.5
connect_models=False

plot_results_by_subject_then_model_formatted(results=results,ci=ci, first_seg=first_seg, final_seg=final_seg, threshold=threshold,
                                             save_fig =save_fig, model_labels=model_labels, connect_models=connect_models,
                                                figwidth=figwidth,figheight=figheight,capsize=capsize, markersize=markersize, edgecolors=edgecolors,linewidth=linewidth)


In [ ]:
# see glm fit for each glm fold for each rat above which are all signif.. 5 rats * 5 folds * 3 models = 75 numbers though
results


In [ ]:
calculate_f2_glm_p_values(results)


In [ ]:
is_first_seg_of_trial = False

first_seg = is_first_seg_of_trial
final_seg = not is_first_seg_of_trial

predictors_a = ['prop_elsewhere_vs_neighbor_leaf']
predictors_c = ['prop_elsewhere_vs_neighbor_leaf_1_ago_bout']
  
if (first_seg == True): #& (final_seg == False):
    data_dict = data_dict_first
elif (first_seg == False) & (final_seg == True):
    data_dict = data_dict_last


In [ ]:
predictionss_a, y_tests_a, d_primes_a, f1_scores_a = plot_switch_roc_stratify_resample_kfold(data_dict, predictors_a, plot_per_subject, subject_ids, save_fig, first_seg, final_seg, is_inner_merge=is_inner_merge)
predictionss_c, y_tests_c, d_primes_c, f1_scores_c = plot_switch_roc_stratify_resample_kfold(data_dict, predictors_c, plot_per_subject, subject_ids, save_fig, first_seg, final_seg, is_inner_merge=is_inner_merge)

In [ ]:
data_dicts = [(predictors_c[0], y_tests_c, predictionss_c),(predictors_a[0], y_tests_a, predictionss_a)] #,(predictors_e[0], y_tests_e, predictionss_e)]
results = process_data(data_dicts, ci)

# balanced_results = process_data_accuracies_cis(data_dicts, ci, use_balanced_accuracy=True)


In [ ]:
# F3

plot_results_by_subject_then_model_formatted(results=results,ci=ci, first_seg=first_seg, final_seg=final_seg, threshold=threshold,
                                             save_fig =save_fig, model_labels=model_labels, connect_models=connect_models,
                                                figwidth=figwidth,figheight=figheight,capsize=capsize, markersize=markersize, 
                                             edgecolors=edgecolors,linewidth=linewidth)

In [ ]:
results

In [ ]:
calculate_f3_glm_p_values(results)